# Install dependencies and environment setup

In [19]:
%pip -qqq install miditoolkit pandarallel kagglehub music21 scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pandarallel import pandarallel
pandarallel.initialize()

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


# Import Dataset from Kaggle

In [7]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

directory = kagglehub.dataset_download(
  "blanderbuss/midi-classic-music"
)

/Users/rogelio/Documents/University_of_San_Diego/AAI_511/aai-511-music-composer-neural-net/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 68.2M/68.2M [00:01<00:00, 43.4MB/s]

Extracting files...


# Restructure and filter midi_files

Filter for Bach, Beethoven, Chopin, and Mozart midi files only

In [12]:
import pandas as pd
import glob

def read_midi_files(composers):
  midi_files = []
  for composer in composers:
    midi_filenames = glob.glob(f'{directory}/midiclassics/{composer}/*.mid')
    midi_filenames += glob.glob(f'{directory}/midiclassics/{composer}/**/*.mid')
    print(f'Found {len(midi_filenames)} .mid files for {composer}')
    midi_files += [{
        'composer': composer,
        'filename': filename
    } for filename in midi_filenames]

  df = pd.DataFrame(midi_files)
  return df

df = read_midi_files(['Bach', 'Beethoven', 'Chopin', 'Mozart'])
df

Found 876 .mid files for Bach
Found 212 .mid files for Beethoven
Found 136 .mid files for Chopin
Found 219 .mid files for Mozart


,composer,filename
0,Bach,/Users/rogelio/.cache/kagglehub/datasets/bland...
1,Bach,/Users/rogelio/.cache/kagglehub/datasets/bland...
2,Bach,/Users/rogelio/.cache/kagglehub/datasets/bland...
3,Bach,/Users/rogelio/.cache/kagglehub/datasets/bland...
4,Bach,/Users/rogelio/.cache/kagglehub/datasets/bland...
...,...,...
1438,Mozart,/Users/rogelio/.cache/kagglehub/datasets/bland...
1439,Mozart,/Users/rogelio/.cache/kagglehub/datasets/bland...
1440,Mozart,/Users/rogelio/.cache/kagglehub/datasets/bland...
1441,Mozart,/Users/rogelio/.cache/kagglehub/datasets/bland...


# Data pre-processing and feature extraction

The `Composer_Dataset` directory contains 3 subdirectories: `dev`, `test`, and `train`. Each subdirectory contains additional subdirectories for each composer, which contain .mid (MIDI) files. The files must be pre-processed to extract relevant features for our deep learning model. The following features may be extracted from the MIDI files:

- Key signature
- Time signature
- Tempo
- Sequence of notes (pitch, duration, velocity)
- Instrumentation
- etc.

In [9]:
from miditoolkit import MidiFile

def read_midi(filename):
    try:
        midi_obj = MidiFile(filename)
        extracted_data = {
            'ticks_per_beat': midi_obj.ticks_per_beat,
            'max_tick': midi_obj.max_tick,
            'tempo_changes_count': len(midi_obj.tempo_changes),
            'time_signature_changes_count': len(midi_obj.time_signature_changes),
            'key_signature_changes_count': len(midi_obj.key_signature_changes),
            'num_instruments': midi_obj.num_instruments,
            'instrument_names': [inst.name for inst in midi_obj.instruments],
            'instruments_data': {
                inst.name: {
                    'is_drum': inst.is_drum,
                    'program': inst.program,
                    'notes': [{
                        'pitch': note.pitch,
                        'start': note.start,
                        'end': note.end,
                        'duration': note.duration,
                        'velocity': note.velocity
                    } for note in inst.notes]
                } for inst in midi_obj.instruments
            }
        }
        return extracted_data
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None

df['midi_data'] = df['filename'].parallel_apply(read_midi)

Error reading /Users/rogelio/.cache/kagglehub/datasets/blanderbuss/midi-classic-music/versions/1/midiclassics/Beethoven/Anhang 14-3.mid: Could not decode key with 3 flats and mode 255


In [10]:
# Remove midi files that cannot be parsed
before_len = len(df)
df = df.dropna()
after_len = len(df)
print(f'Removed {before_len - after_len} midi file(s) that cannot be parsed')

Removed 1 midi file(s) that cannot be parsed


### DATA PRE-PROCESSING, MIDI CONVERSION, AUGMENTATION, AND EVENT FEATURE EXTRACTION
Symbolic-music models commonly represent a score as an ordered sequence of note
events. MusicBERT's OctupleMIDI representation demonstrates the usefulness of
metrical position, instrument, pitch, duration, velocity, tempo, and time
signature for symbolic-music understanding (Zeng et al., 2021). Accordingly,
this cell extracts a compact event representation containing pitch, pitch
class, duration, velocity, onset spacing, instrument program, tempo, and meter.

Pitch transposition and tempo scaling are used as label-preserving training
augmentations. They increase tonal and temporal variation while retaining the
composer label. Augmentation must be applied only after creating grouped
train/validation/test splits; all versions sharing the same ``source_id`` must
remain in the same split to prevent data leakage.

### APA 7 references
Cuthbert, M. S., & Ariza, C. (2010). music21: A toolkit for
    computer-aided musicology and symbolic music data. In J. S. Downie &
    R. C. Veltkamp (Eds.), Proceedings of the 11th International Society
    for Music Information Retrieval Conference (pp. 637–642).

McLeod, A., Owers, J., & Yoshii, K. (2020). The MIDI degradation toolkit:
    Symbolic music augmentation and correction. arXiv.
    https://doi.org/10.48550/arXiv.2010.00059

Zeng, M., Tan, X., Wang, R., Ju, Z., Qin, T., & Liu, T.-Y. (2021).
    MusicBERT: Symbolic music understanding with large-scale pre-training.
    In Findings of the Association for Computational Linguistics:
    ACL-IJCNLP 2021 (pp. 791–800).
    https://doi.org/10.18653/v1/2021.findings-acl.70

In [16]:
from __future__ import annotations

from bisect import bisect_right
from copy import deepcopy
from pathlib import Path
from typing import Iterable
import hashlib
import numpy as np
import pandas as pd
from miditoolkit import MidiFile

# music21 is used only when a source score is not already MIDI.
from music21 import converter


PREPROCESSED_DIR = Path("preprocessed_midi")
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Conservative transformations help preserve recognizable compositional style.
PITCH_SHIFTS = (-2, 2)          # semitones
TEMPO_FACTORS = (0.90, 1.10)    # playback-rate multipliers
MAX_EVENTS = 2048               # fixed upper bound for CNN/LSTM sequences


def stable_source_id(path: str | Path) -> str:
    """Return a stable ID used to group original and augmented versions."""
    return hashlib.sha1(str(Path(path).resolve()).encode("utf-8")).hexdigest()[:16]


def convert_score_to_midi(score_path: str | Path,
                          output_dir: Path = PREPROCESSED_DIR) -> Path:
    """
    Convert MusicXML, MXL, ABC, or another music21-readable score to MIDI.
    Existing MIDI files are returned unchanged.
    """
    score_path = Path(score_path)
    if score_path.suffix.lower() in {".mid", ".midi"}:
        return score_path

    output_path = output_dir / f"{score_path.stem}.mid"
    parsed_score = converter.parse(str(score_path))
    parsed_score.write("midi", fp=str(output_path))
    return output_path


def augment_midi(midi_path: str | Path,
                 pitch_shift: int = 0,
                 tempo_factor: float = 1.0,
                 output_dir: Path = PREPROCESSED_DIR) -> Path:
    """
    Create an augmented MIDI file.

    Pitch shift changes non-drum note pitches. Tempo scaling modifies tempo
    events without changing the notated tick positions. Invalid MIDI pitches
    outside 0–127 are clipped.
    """
    midi_path = Path(midi_path)
    midi = MidiFile(str(midi_path))

    for instrument in midi.instruments:
        if not instrument.is_drum and pitch_shift != 0:
            for note in instrument.notes:
                note.pitch = int(np.clip(note.pitch + pitch_shift, 0, 127))

    if tempo_factor != 1.0:
        for change in midi.tempo_changes:
            change.tempo = float(np.clip(change.tempo * tempo_factor, 20.0, 300.0))

    suffix = f"_ps{pitch_shift:+d}_tf{tempo_factor:.2f}".replace(".", "p")
    output_path = output_dir / f"{midi_path.stem}{suffix}.mid"
    midi.dump(str(output_path))
    return output_path


def _active_tempo(tick: int, tempo_ticks: list[int], tempo_values: list[float]) -> float:
    """Return the most recent tempo at a note onset."""
    idx = max(0, bisect_right(tempo_ticks, tick) - 1)
    return tempo_values[idx]


def _active_meter(tick: int,
                  meter_ticks: list[int],
                  numerators: list[int],
                  denominators: list[int]) -> tuple[int, int]:
    """Return the most recent time signature at a note onset."""
    idx = max(0, bisect_right(meter_ticks, tick) - 1)
    return numerators[idx], denominators[idx]


def extract_event_features(midi_path: str | Path,
                           max_events: int = MAX_EVENTS) -> dict:
    """
    Convert a MIDI file into a chronological sequence of event-level features.

    Each event contains:
      pitch, pitch_class, duration_beats, velocity, onset_delta_beats,
      program, tempo_bpm, time_signature_numerator, and
      time_signature_denominator.
    """
    midi = MidiFile(str(midi_path))
    tpq = max(int(midi.ticks_per_beat), 1)

    tempo_changes = sorted(midi.tempo_changes, key=lambda x: x.time)
    tempo_ticks = [int(x.time) for x in tempo_changes] or [0]
    tempo_values = [float(x.tempo) for x in tempo_changes] or [120.0]

    meter_changes = sorted(midi.time_signature_changes, key=lambda x: x.time)
    meter_ticks = [int(x.time) for x in meter_changes] or [0]
    numerators = [int(x.numerator) for x in meter_changes] or [4]
    denominators = [int(x.denominator) for x in meter_changes] or [4]

    raw_events = []
    for instrument in midi.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            raw_events.append((
                int(note.start), int(note.end), int(note.pitch),
                int(note.velocity), int(instrument.program)
            ))

    raw_events.sort(key=lambda e: (e[0], e[2], e[1]))
    raw_events = raw_events[:max_events]

    features = []
    previous_start = raw_events[0][0] if raw_events else 0
    for start, end, pitch, velocity, program in raw_events:
        numerator, denominator = _active_meter(
            start, meter_ticks, numerators, denominators
        )
        features.append({
            "pitch": pitch,
            "pitch_class": pitch % 12,
            "duration_beats": max(end - start, 1) / tpq,
            "velocity": velocity / 127.0,
            "onset_delta_beats": max(start - previous_start, 0) / tpq,
            "program": program,
            "tempo_bpm": _active_tempo(start, tempo_ticks, tempo_values),
            "time_signature_numerator": numerator,
            "time_signature_denominator": denominator,
        })
        previous_start = start

    return {
        "event_features": features,
        "sequence_length": len(features),
        "ticks_per_beat": tpq,
    }


def build_preprocessed_dataset(
    source_df: pd.DataFrame,
    pitch_shifts: Iterable[int] = PITCH_SHIFTS,
    tempo_factors: Iterable[float] = TEMPO_FACTORS,
    include_augmentations: bool = True,
) -> pd.DataFrame:
    """
    Convert files to MIDI, create augmentations, and extract model-ready events.

    The resulting ``source_id`` must be used as the grouping variable when
    splitting the data. Ideally, first create grouped splits and call this
    function with augmentations enabled only for the training subset.
    """
    records = []

    for row in source_df[["composer", "filename"]].itertuples(index=False):
        try:
            midi_path = convert_score_to_midi(row.filename)
            source_id = stable_source_id(row.filename)

            variants = [("original", midi_path, 0, 1.0)]
            if include_augmentations:
                variants += [
                    (f"pitch_{shift:+d}", augment_midi(midi_path, pitch_shift=shift),
                     shift, 1.0)
                    for shift in pitch_shifts
                ]
                variants += [
                    (f"tempo_{factor:.2f}", augment_midi(midi_path, tempo_factor=factor),
                     0, factor)
                    for factor in tempo_factors
                ]

            for augmentation, variant_path, shift, factor in variants:
                extracted = extract_event_features(variant_path)
                records.append({
                    "composer": row.composer,
                    "source_id": source_id,
                    "augmentation": augmentation,
                    "pitch_shift": shift,
                    "tempo_factor": factor,
                    "midi_path": str(variant_path),
                    **extracted,
                })
        except Exception as exc:
            print(f"Skipped {row.filename}: {exc}")

    return pd.DataFrame(records)

In [17]:

# For a leakage-safe final experiment, split df by source file/composer first and
# run augmentation only on the training partition. This call demonstrates the
# complete preprocessing pipeline required by the project.
processed_df = build_preprocessed_dataset(df, include_augmentations=True)

print(f"Created {len(processed_df):,} original/augmented sequences.")
display(
    processed_df[
        ["composer", "source_id", "augmentation", "sequence_length", "midi_path"]
    ].head()
)
print("\nClass and augmentation counts:")
display(pd.crosstab(processed_df["composer"], processed_df["augmentation"]))

Skipped /Users/rogelio/.cache/kagglehub/datasets/blanderbuss/midi-classic-music/versions/1/midiclassics/Beethoven/Anhang 14-3.mid: Could not decode key with 3 flats and mode 255
Created 7,210 original/augmented sequences.


,composer,source_id,augmentation,sequence_length,midi_path
0,Bach,b92097a6e97bf055,original,977,/Users/rogelio/.cache/kagglehub/datasets/bland...
1,Bach,b92097a6e97bf055,pitch_-2,977,preprocessed_midi/Bwv0997 Partita for Lute 1mo...
2,Bach,b92097a6e97bf055,pitch_+2,977,preprocessed_midi/Bwv0997 Partita for Lute 1mo...
3,Bach,b92097a6e97bf055,tempo_0.90,977,preprocessed_midi/Bwv0997 Partita for Lute 1mo...
4,Bach,b92097a6e97bf055,tempo_1.10,977,preprocessed_midi/Bwv0997 Partita for Lute 1mo...



Class and augmentation counts:


augmentation,original,pitch_+2,pitch_-2,tempo_0.90,tempo_1.10
composer,,,,,
Bach,876,876,876,876,876
Beethoven,211,211,211,211,211
Chopin,136,136,136,136,136
Mozart,219,219,219,219,219


### Dataset formatting, leakage-safe splitting, and NumPy serialization

This section converts the event dictionaries in `processed_df` into fixed-size numerical tensors for CNN/LSTM models. Splits are performed at the **original source-file level** using `source_id`, with stratification by composer. All augmented variants of a source are restricted to the training set, while validation and test sets contain only original MIDI sequences. This prevents near-duplicate augmented examples from leaking across partitions and producing overly optimistic evaluation results.

Generated artifacts:

- `X_train`, `X_val`, `X_test`: event tensors with shape `(samples, MAX_EVENTS, features)`
- `mask_train`, `mask_val`, `mask_test`: valid-event masks
- `y_train`, `y_val`, `y_test`: integer composer labels
- compressed `.npz` split files, a label map, a feature specification, and CSV manifests

In [21]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Reproducible split configuration.
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

# Fixed feature order used by every split and by downstream models.
FEATURE_NAMES = [
    "pitch",
    "pitch_class",
    "duration_beats",
    "velocity",
    "onset_delta_beats",
    "program",
    "tempo_bpm",
    "time_signature_numerator",
    "time_signature_denominator",
]

SERIALIZED_DIR = Path("serialized_dataset")
SERIALIZED_DIR.mkdir(parents=True, exist_ok=True)


def validate_processed_dataframe(dataframe: pd.DataFrame) -> None:
    """Check that preprocessing created the columns required below."""
    required = {"composer", "source_id", "augmentation", "event_features"}
    missing = required.difference(dataframe.columns)
    if missing:
        raise ValueError(
            "processed_df is missing required columns: " + ", ".join(sorted(missing))
        )
    if dataframe.empty:
        raise ValueError("processed_df is empty; run the preprocessing cell first.")


def make_source_level_splits(dataframe: pd.DataFrame):
    """
    Split unique original sources with composer stratification.

    Augmented rows are deliberately excluded when deriving source-level labels.
    """
    source_table = (
        dataframe[["source_id", "composer"]]
        .drop_duplicates()
        .sort_values(["composer", "source_id"])
        .reset_index(drop=True)
    )

    # Every source must map to exactly one composer.
    conflicts = source_table.groupby("source_id")["composer"].nunique()
    if (conflicts > 1).any():
        bad_ids = conflicts[conflicts > 1].index.tolist()[:5]
        raise ValueError(f"Some source IDs have multiple composer labels: {bad_ids}")

    class_counts = source_table["composer"].value_counts()
    if (class_counts < 3).any():
        raise ValueError(
            "Each composer needs at least three original files for train/validation/test "
            f"splitting. Counts: {class_counts.to_dict()}"
        )

    train_sources, temp_sources = train_test_split(
        source_table,
        train_size=TRAIN_SIZE,
        random_state=RANDOM_STATE,
        stratify=source_table["composer"],
    )

    # 15% validation and 15% test means splitting the remaining 30% equally.
    relative_test_size = TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE)
    val_sources, test_sources = train_test_split(
        temp_sources,
        test_size=relative_test_size,
        random_state=RANDOM_STATE,
        stratify=temp_sources["composer"],
    )

    return train_sources, val_sources, test_sources


def select_rows_for_split(
    dataframe: pd.DataFrame,
    source_table: pd.DataFrame,
    include_augmentations: bool,
) -> pd.DataFrame:
    """Select rows by source ID and optionally retain augmented variants."""
    selected = dataframe[dataframe["source_id"].isin(source_table["source_id"])].copy()
    if not include_augmentations:
        selected = selected[selected["augmentation"] == "original"].copy()

    return selected.sort_values(
        ["composer", "source_id", "augmentation"]
    ).reset_index(drop=True)


def events_to_tensor(
    event_sequences,
    max_events: int = MAX_EVENTS,
    feature_names=FEATURE_NAMES,
):
    """
    Convert variable-length event dictionaries to a padded float32 tensor.

    Padding rows contain zeros. The mask distinguishes padding from real events.
    Selected continuous features are scaled to numerically stable ranges using
    fixed MIDI/music-domain constants rather than statistics from validation/test.
    """
    n_samples = len(event_sequences)
    n_features = len(feature_names)
    X = np.zeros((n_samples, max_events, n_features), dtype=np.float32)
    mask = np.zeros((n_samples, max_events), dtype=np.bool_)

    # Fixed, interpretable scaling. pitch_class is already bounded by 0–11.
    scale = {
        "pitch": 127.0,
        "pitch_class": 11.0,
        "duration_beats": 16.0,
        "velocity": 1.0,  # already normalized during feature extraction
        "onset_delta_beats": 16.0,
        "program": 127.0,
        "tempo_bpm": 300.0,
        "time_signature_numerator": 16.0,
        "time_signature_denominator": 16.0,
    }

    for sample_index, events in enumerate(event_sequences):
        if not isinstance(events, list):
            continue
        usable_events = events[:max_events]
        for event_index, event in enumerate(usable_events):
            X[sample_index, event_index] = [
                np.clip(float(event.get(name, 0.0)) / scale[name], 0.0, 1.0)
                for name in feature_names
            ]
        mask[sample_index, :len(usable_events)] = True

    return X, mask


def encode_and_format(split_df: pd.DataFrame, label_to_index: dict):
    """Create X, mask, and integer y arrays for one dataframe split."""
    X, mask = events_to_tensor(split_df["event_features"].tolist())
    y = split_df["composer"].map(label_to_index).to_numpy(dtype=np.int64)
    if np.isnan(y.astype(float)).any():
        raise ValueError("At least one composer could not be encoded.")
    return X, mask, y


def save_split(name, X, mask, y, manifest: pd.DataFrame):
    """Serialize arrays as compressed NPZ and metadata as CSV."""
    np.savez_compressed(
        SERIALIZED_DIR / f"{name}.npz",
        X=X,
        mask=mask,
        y=y,
        feature_names=np.asarray(FEATURE_NAMES),
    )
    manifest_columns = [
        column for column in
        ["composer", "source_id", "augmentation", "midi_path", "sequence_length"]
        if column in manifest.columns
    ]
    manifest[manifest_columns].to_csv(
        SERIALIZED_DIR / f"{name}_manifest.csv", index=False
    )

In [22]:
validate_processed_dataframe(processed_df)
train_sources, val_sources, test_sources = make_source_level_splits(processed_df)

# Training includes augmentations. Validation/test include original sources only.
train_df = select_rows_for_split(processed_df, train_sources, include_augmentations=True)
val_df = select_rows_for_split(processed_df, val_sources, include_augmentations=False)
test_df = select_rows_for_split(processed_df, test_sources, include_augmentations=False)

# Deterministic alphabetical encoding supports reproducibility.
composer_names = sorted(processed_df["composer"].unique().tolist())
label_to_index = {composer: index for index, composer in enumerate(composer_names)}
index_to_label = {index: composer for composer, index in label_to_index.items()}

X_train, mask_train, y_train = encode_and_format(train_df, label_to_index)
X_val, mask_val, y_val = encode_and_format(val_df, label_to_index)
X_test, mask_test, y_test = encode_and_format(test_df, label_to_index)

save_split("train", X_train, mask_train, y_train, train_df)
save_split("validation", X_val, mask_val, y_val, val_df)
save_split("test", X_test, mask_test, y_test, test_df)

# Save metadata needed to decode predictions and reproduce preprocessing.
with open(SERIALIZED_DIR / "label_map.json", "w", encoding="utf-8") as file:
    json.dump(
        {"label_to_index": label_to_index, "index_to_label": index_to_label},
        file,
        indent=2,
    )

with open(SERIALIZED_DIR / "dataset_specification.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "random_state": RANDOM_STATE,
            "split_ratios": {
                "train": TRAIN_SIZE,
                "validation": VALIDATION_SIZE,
                "test": TEST_SIZE,
            },
            "max_events": MAX_EVENTS,
            "feature_names": FEATURE_NAMES,
            "tensor_dtype": "float32",
            "label_dtype": "int64",
            "padding_value": 0.0,
            "training_uses_augmentations": True,
            "validation_and_test_use_originals_only": True,
        },
        file,
        indent=2,
    )

# Confirm that no original source appears in more than one partition.
train_ids = set(train_sources["source_id"])
val_ids = set(val_sources["source_id"])
test_ids = set(test_sources["source_id"])
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

summary = pd.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "unique_sources": [
            train_df["source_id"].nunique(),
            val_df["source_id"].nunique(),
            test_df["source_id"].nunique(),
        ],
        "tensor_shape": [X_train.shape, X_val.shape, X_test.shape],
        "array_size_mb": [
            round((X_train.nbytes + mask_train.nbytes + y_train.nbytes) / 1024**2, 2),
            round((X_val.nbytes + mask_val.nbytes + y_val.nbytes) / 1024**2, 2),
            round((X_test.nbytes + mask_test.nbytes + y_test.nbytes) / 1024**2, 2),
        ],
    }
)

print("Composer label mapping:", label_to_index)
print(f"Serialized files written to: {SERIALIZED_DIR.resolve()}")
display(summary)
print("\nClass distribution by split:")
display(
    pd.concat(
        {
            "Train": train_df["composer"].value_counts(),
            "Validation": val_df["composer"].value_counts(),
            "Test": test_df["composer"].value_counts(),
        },
        axis=1,
    ).fillna(0).astype(int)
)

Composer label mapping: {'Bach': 0, 'Beethoven': 1, 'Chopin': 2, 'Mozart': 3}
Serialized files written to: /Users/rogelio/Documents/University_of_San_Diego/AAI_511/aai-511-music-composer-neural-net/serialized_dataset


,split,rows,unique_sources,tensor_shape,array_size_mb
0,Train,5045,1009,"(5045, 2048, 9)",364.62
1,Validation,216,216,"(216, 2048, 9)",15.61
2,Test,217,217,"(217, 2048, 9)",15.68



Class distribution by split:


,Train,Validation,Test
composer,,,
Bach,3065,131,132
Mozart,765,33,33
Beethoven,740,31,32
Chopin,475,21,20


# Model Building

Define model architecture utilizing CNN and LSTMs. Reference existing research papers and articles for inspiration.

# Model Training

Train the model using the processed dataset.

# Model Evaluation

Evaluate model performance using the following performance metrics:

- Accuracy
- Precision
- Recall
- F1-score
- AUC-ROC

Use the following visualization techniques to analyze the model's performance:
- Plot training and validation loss curves
- Plot confusion matrix

# Model Optimization

Optimize the model using hyperparameter tuning.